# Missão Aurora Siger: Verificação de segurança pré-decolagem
---
**Autores:** André Raposo (RM568837), Google Gemini  

## 1. Geração de dados regulares
Inícia a geração de dados de telemetria utilizando as bibliotecas `pandas` e `numpy`. Considera um cenário ideal onde todos os dados estão dentro da faixa de limites esperada.
 

In [2]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

n_rows = 500
start_time = datetime.now()

# Gera base de dados com valores randomicos dentro do limite.
data = {
    "timestamp": [start_time + timedelta(minutes=i) for i in range(n_rows)],
    "internal_temp_c": np.random.uniform(18, 35, n_rows),
    "external_temp_c": np.random.uniform(-5, 30, n_rows),
    "battery_voltage_v": np.random.uniform(46, 52, n_rows),
    "battery_current_a": np.random.uniform(20, 120, n_rows),
    "battery_soc_percentage": np.random.uniform(60, 100, n_rows),
    "battery_capacity_ah": np.random.uniform(80, 120, n_rows),
    "power_load_kw": np.random.uniform(5, 25, n_rows),
    "energy_loss_percent": np.random.uniform(2, 8, n_rows),
    "tank_pressure_bar": np.random.uniform(95, 145, n_rows),
    "estimated_autonomy_min": np.random.uniform(45, 180, n_rows),
    "structural_integrity": 1,
    "critical_modules_status": 1,
    "telemetry_link_status": 1,
    "is_anomaly": False
}

# Cálcula energia disponível
data["energy_available_kwh"] = (data["battery_voltage_v"] * data["battery_capacity_ah"] * (data["battery_soc_percentage"]/100)) / 1000

# Carrega os dados em um dataframe
df = pd.DataFrame(data)

## 2. Geração de dados com anomalias
Introduz um grau de 5% de anomalias nos dados gerados anteriormente

---
O trecho de código foi parcialmente criado utilizando o Google Gemini com o seguinte prompt:   

> "Com base nos dados gerados anteriormente introduza anomalia nos dados de telemetria
>
> Regras: 
> - Insira anomalia em 5% dos dados 
> - Temperatura interna entre 45 e 70 
> - Nível de bateria entre 5 e 25 
> - Pressão fora da faixa (40-80 ou 160-220) 
> - Integridade estrutural 0 
> - Estado dos módulos críticos 0 
> - Estados link de telemetria 0 
>
> Saída 
> - Arquivo CSV válido"

O código foi avaliado e refatorado para remover "alucionações" e melhorar a legibilidade:   


**Alterações:**
- Um dos tipos de anomalia estava com um erro no nome
- Utiliza enum para definir tipos de anomalia `AnomalyTypes` para evitar possíveis erros de digitação.

In [3]:
# Define a porcentagem de anomalia nos dados: 5%
from enum import Enum
from typing import List


n_anomalies = int(len(df) * 0.05)

# Busca por indices aleatóriamente para introduzir anomalias
anomaly_idx = np.random.choice(df.index, n_anomalies, replace=False)


# Define os tipos de dados que poderão ter anomalias
# Define Anomaly Types using an Enum
class AnomalyTypes(Enum):
    INTERNAL_TEMP = 'internal_temp_c'
    BATTERY_SOC = 'battery_soc_percentage'
    TANK_PRESSURE = 'tank_pressure_bar'
    STRUCTURAL_INTEGRITY = 'structural_integrity'
    CRITICAL_MODULES_STATUS = 'critical_modules_status'
    TELEMETRY_LINK_STATUS = 'telemetry_link_status'

anomaly_types: List = list(AnomalyTypes)

for idx in anomaly_idx:
    df.loc[idx, "is_anomaly"] = True
    anomaly_type = np.random.choice(anomaly_types)
    print(anomaly_type.value)

    match anomaly_type:
        case AnomalyTypes.INTERNAL_TEMP:
            df.loc[idx, anomaly_type.value] = np.random.uniform(45, 70)
        case AnomalyTypes.BATTERY_SOC:
            df.loc[idx, anomaly_type.value] = np.random.uniform(5, 25)
        case AnomalyTypes.TANK_PRESSURE:
            if np.random.rand() > 0.5:
                df.loc[idx, anomaly_type.value] = np.random.uniform(40, 80)
            else:
                df.loc[idx, anomaly_type.value] = np.random.uniform(160, 220)
        case AnomalyTypes.STRUCTURAL_INTEGRITY:
            df.loc[idx, anomaly_type.value] = 0
        case AnomalyTypes.CRITICAL_MODULES_STATUS:
            df.loc[idx, anomaly_type.value] = 0
        case AnomalyTypes.TELEMETRY_LINK_STATUS:
            df.loc[idx, anomaly_type.value] = 0


df.to_csv("telemetria_simulada.csv", index=False)
df.head()

critical_modules_status
telemetry_link_status
tank_pressure_bar
structural_integrity
internal_temp_c
critical_modules_status
battery_soc_percentage
internal_temp_c
critical_modules_status
battery_soc_percentage
structural_integrity
tank_pressure_bar
tank_pressure_bar
internal_temp_c
battery_soc_percentage
telemetry_link_status
battery_soc_percentage
structural_integrity
critical_modules_status
tank_pressure_bar
battery_soc_percentage
critical_modules_status
telemetry_link_status
tank_pressure_bar
structural_integrity


,timestamp,internal_temp_c,external_temp_c,battery_voltage_v,battery_current_a,battery_soc_percentage,battery_capacity_ah,power_load_kw,energy_loss_percent,tank_pressure_bar,estimated_autonomy_min,structural_integrity,critical_modules_status,telemetry_link_status,is_anomaly,energy_available_kwh
0,2026-03-21 19:44:05.387923,21.291579,11.857457,50.793309,66.039468,85.123926,91.575665,8.726949,6.078936,129.745963,132.128731,1,1,1,False,3.959481
1,2026-03-21 19:45:05.387923,19.197657,-1.740124,48.765791,28.659636,63.001954,97.426115,11.465771,4.372988,101.597719,54.274694,1,1,1,False,2.993262
2,2026-03-21 19:46:05.387923,27.815247,12.509207,46.855634,107.969003,62.371108,106.108001,24.321042,4.292566,110.071463,64.253490,1,1,1,False,3.100940
3,2026-03-21 19:47:05.387923,32.250674,21.202989,48.069552,64.295905,81.352119,114.564983,15.350981,3.231022,119.097623,161.895903,1,1,1,False,4.480132
4,2026-03-21 19:48:05.387923,29.385805,17.340882,47.679743,108.869236,84.727102,119.232566,14.907844,6.875807,127.140548,62.002383,1,1,1,False,4.816717


## 3. Verificação Pre-decolagem
Realiza a verificação de segurança utilizando os dados simulados.

In [4]:
from verificacao_pre_decolagem import pronto_para_decolar

df_result = df.copy()

# Itera por cada linha do dataframe
for idx, row in df.iterrows():
    df_result.loc[idx, "is_ready_to_launch"] = pronto_para_decolar(row.to_dict())

df_result.to_csv("telemetria_resultado.csv", index=False)
df_result.head()

,timestamp,internal_temp_c,external_temp_c,battery_voltage_v,battery_current_a,battery_soc_percentage,battery_capacity_ah,power_load_kw,energy_loss_percent,tank_pressure_bar,estimated_autonomy_min,structural_integrity,critical_modules_status,telemetry_link_status,is_anomaly,energy_available_kwh,is_ready_to_launch
0,2026-03-21 19:44:05.387923,21.291579,11.857457,50.793309,66.039468,85.123926,91.575665,8.726949,6.078936,129.745963,132.128731,1,1,1,False,3.959481,True
1,2026-03-21 19:45:05.387923,19.197657,-1.740124,48.765791,28.659636,63.001954,97.426115,11.465771,4.372988,101.597719,54.274694,1,1,1,False,2.993262,True
2,2026-03-21 19:46:05.387923,27.815247,12.509207,46.855634,107.969003,62.371108,106.108001,24.321042,4.292566,110.071463,64.253490,1,1,1,False,3.100940,True
3,2026-03-21 19:47:05.387923,32.250674,21.202989,48.069552,64.295905,81.352119,114.564983,15.350981,3.231022,119.097623,161.895903,1,1,1,False,4.480132,True
4,2026-03-21 19:48:05.387923,29.385805,17.340882,47.679743,108.869236,84.727102,119.232566,14.907844,6.875807,127.140548,62.002383,1,1,1,False,4.816717,True


# 4. Teste do algoritmo de verificação de pré-decolagem
Visto que os dados com anomalia estão identificados com a flag `is_anomaly`, podemos utilizá-la para assegurar que todos os casos de anomalia foram capturados pelo algorítmo de verificação.

A verificação é feita da seguinte forma:
1. Filtra-se o dataframe de resultados por linhas onde `is_anomaly` é verdadeiro e `is_ready_to_launch` é verdadeiro.
2. Verifica-se se o dataframe filtrado está vazio indicando que não há possibilidade de lançamento em casos de anomalia.
3. Realiza a verificação oposta filtrando por linhas onde `is_anomaly` é falso e `is_ready_to_launch` é falso.
4. Verifica se o dataframe resultante está vazio indicando que não há possibilidade de abortar um lançamento quando não há anomalias.

In [5]:
# Filtra o dataframe por linhas com anomalia e prontas para decolagem
df_anomalies = df_result[(df_result["is_anomaly"] == True) & (df_result["is_ready_to_launch"] == True)]

# Assegura que o número de decolagens permitidas em dados com anomalia é igual a 0
assert(len(df_anomalies) == 0)

# Filtra o dataframe por linhas sem anomalia e com decolagem abortada
df_no_anomalies = df_result[(df_result["is_anomaly"] == False) & (df_result["is_ready_to_launch"] == False)]

# Assegura que o número de decolagens abortadas em dados sem anomalia é igual a 0
assert(len(df_no_anomalies) == 0)
